# Convert Neutrino Detection Model to ONNX

Converts the trained PyTorch model to ONNX format for deployment.

In [10]:
import torch
import torch.onnx
import onnx
import onnxruntime as ort
import yaml
import numpy as np
from src.base_models import create_model

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda:0


In [11]:
# Load model
with open('model_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

model = create_model(config['model'])
model.load_state_dict(torch.load('base_model.pth', map_location=device))
model.to(device)
model.eval()

print(f"Model loaded: {config['experiment_name']}")
print(f"Parameters: {config['parameters']:,}")

Model loaded: da_numu_251108_middle_constlambda_aug
Parameters: 541,633


/tmp/ipykernel_483220/1999866903.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('base_model.pth', map_location=device))


In [12]:
# Create sample input
batch_size = 1
seq_len = 100
input_dim = 5

# Create dummy input data
features = torch.randn(batch_size, seq_len, input_dim, device=device)
lengths = torch.tensor([seq_len], device=device)
mask = torch.ones(batch_size, seq_len, dtype=torch.bool, device=device)

sample_batch = {
    'features': features,
    'lengths': lengths,
    'mask': mask
}

print(f"Input shapes: features={features.shape}, lengths={lengths.shape}, mask={mask.shape}")

Input shapes: features=torch.Size([1, 100, 5]), lengths=torch.Size([1]), mask=torch.Size([1, 100])


In [13]:
# Test model
with torch.no_grad():
    output = model(sample_batch)
    prob = torch.sigmoid(output)

print(f"Model output shape: {output.shape}")
print(f"Model works correctly: ✅")

Model output shape: torch.Size([1, 1])
Model works correctly: ✅


In [14]:
# Create ONNX wrapper
class ONNXWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, features, lengths, mask):
        batch = {'features': features, 'lengths': lengths, 'mask': mask}
        return self.model(batch)

onnx_model = ONNXWrapper(model)
onnx_model.eval()

# Verify wrapper works
with torch.no_grad():
    wrapper_output = onnx_model(features, lengths, mask)
    
assert torch.allclose(output, wrapper_output, atol=1e-6)
print("ONNX wrapper created and verified: ✅")

ONNX wrapper created and verified: ✅


In [18]:
# Export to ONNX
onnx_path = 'base_model.onnx'

torch.onnx.export(
    onnx_model,
    (features, lengths, mask),
    onnx_path,
    export_params=True,
    opset_version=17,
    input_names=['features', 'lengths', 'mask'],
    output_names=['logits'],
    dynamic_axes={
        'features': {0: 'batch_size', 1: 'sequence_length'},
        'lengths': {0: 'batch_size'},
        'mask': {0: 'batch_size', 1: 'sequence_length'},
        'logits': {0: 'batch_size'}
    }
)

print(f"Model exported to: {onnx_path}")

Model exported to: base_model.onnx


In [19]:
# Verify ONNX model
onnx_model_check = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model_check)

# Check actual input names
print("ONNX model inputs:")
for inp in onnx_model_check.graph.input:
    print(f"  {inp.name}: {[d.dim_value if d.dim_value > 0 else 'dynamic' for d in inp.type.tensor_type.shape.dim]}")

# Test with ONNX Runtime
ort_session = ort.InferenceSession(onnx_path)

print("\nONNX Runtime inputs:")
for inp in ort_session.get_inputs():
    print(f"  {inp.name}: {inp.shape}")

# Create inputs based on actual ONNX model inputs
ort_input_names = [inp.name for inp in ort_session.get_inputs()]
ort_inputs = {}

if 'features' in ort_input_names:
    ort_inputs['features'] = features.cpu().numpy()
if 'lengths' in ort_input_names:
    ort_inputs['lengths'] = lengths.cpu().numpy()
if 'mask' in ort_input_names:
    ort_inputs['mask'] = mask.cpu().numpy()

print(f"\nUsing inputs: {list(ort_inputs.keys())}")

ort_outputs = ort_session.run(None, ort_inputs)
onnx_logits = ort_outputs[0]

# Compare results
pytorch_logits = output.cpu().numpy()
diff = np.abs(pytorch_logits - onnx_logits).max()
match = np.allclose(pytorch_logits, onnx_logits, atol=1e-5)

print(f"\nPyTorch vs ONNX max difference: {diff:.2e}")
print(f"Results match: {'✅' if match else '❌'}")

ONNX model inputs:
  features: ['dynamic', 'dynamic', 5]
  mask: ['dynamic', 'dynamic']

ONNX Runtime inputs:
  features: ['batch_size', 'sequence_length', 5]
  mask: ['batch_size', 'sequence_length']

Using inputs: ['features', 'mask']

PyTorch vs ONNX max difference: 0.00e+00
Results match: ✅
